In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px
import kaleido

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
    learning_curve,
    validation_curve
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

# Models

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

# Feature Selection

from sklearn.feature_selection import (
    SelectKBest,
    mutual_info_classif,
    RFE
)

# Metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

# Permutation Importance

from sklearn.inspection import permutation_importance

# Save Models

import joblib

# Silence warnings

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/Telco-Customer-Churn.csv")

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.shape

(7043, 21)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [6]:
df.describe(include="all")

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
count,7043,7043,7043.000000,7043,7043,7043.000000,7043,7043,7043,7043,...,7043,7043,7043,7043,7043,7043,7043,7043.000000,7043,7043
unique,7043,2,NaN,2,2,NaN,2,3,3,3,...,3,3,3,3,3,2,4,NaN,6531,2
top,7590-VHVEG,Male,NaN,No,No,NaN,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,NaN,20.2,No
freq,1,3555,NaN,3641,4933,NaN,6361,3390,3096,3498,...,3095,3473,2810,2785,3875,4171,2365,NaN,11,5174
mean,NaN,NaN,0.162147,NaN,NaN,32.371149,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.761692,NaN,NaN
std,NaN,NaN,0.368612,NaN,NaN,24.559481,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.090047,NaN,NaN
min,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.250000,NaN,NaN
25%,NaN,NaN,0.000000,NaN,NaN,9.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.500000,NaN,NaN
50%,NaN,NaN,0.000000,NaN,NaN,29.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.350000,NaN,NaN
75%,NaN,NaN,0.000000,NaN,NaN,55.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.850000,NaN,NaN


In [7]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

## Data Cleaning

In [8]:
df.isnull().sum().sort_values(ascending=False)

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [9]:
(df == "").sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [10]:
(df == " ").sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [11]:
df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan)

In [12]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [14]:
df.isnull().sum().sort_values(ascending=False)

TotalCharges        11
gender               0
SeniorCitizen        0
Partner              0
customerID           0
Dependents           0
tenure               0
MultipleLines        0
PhoneService         0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
InternetService      0
TechSupport          0
StreamingTV          0
Contract             0
StreamingMovies      0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
Churn                0
dtype: int64

In [15]:
df = df.dropna()

In [16]:
df.shape

(7032, 21)

In [17]:
df.duplicated().sum()

np.int64(0)

In [18]:
df["Churn"].value_counts()

Churn
No     5163
Yes    1869
Name: count, dtype: int64

In [19]:
df["Churn"].value_counts(normalize=True)

Churn
No     0.734215
Yes    0.265785
Name: proportion, dtype: float64

## Feature Engineering & Preprocessing

In [20]:
#drop
df = df.drop(columns=["customerID"])

In [21]:
#encode
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [22]:
df["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [23]:
#feature / target
X = df.drop(columns=["Churn"])

y = df["Churn"]

In [24]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

numeric_features

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str')

In [25]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns

categorical_features

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='str')

In [26]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [27]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)
# handel unkown ignore: allows the model to handle unseen categories during prediction without failing

In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [30]:
print(X_train.shape)
print(X_test.shape)

print()

print(y_train.value_counts(normalize=True))

print()

print(y_test.value_counts(normalize=True))

(5625, 19)
(1407, 19)

Churn
0    0.734222
1    0.265778
Name: proportion, dtype: float64

Churn
0    0.734186
1    0.265814
Name: proportion, dtype: float64


## Baseline Models

In [31]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=42,
                max_iter=1000
            )
        )
    ]
)

In [32]:
tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

In [33]:
forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                random_state=42
            )
        )
    ]
)

In [34]:
gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            GradientBoostingClassifier(
                random_state=42
            )
        )
    ]
)

In [35]:
models = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": tree_pipeline,
    "Random Forest": forest_pipeline,
    "Gradient Boosting": gb_pipeline
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1 Score": f1_score(y_test, predictions),
        "ROC AUC": roc_auc_score(y_test, probabilities)
    })

In [36]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="ROC AUC",
    ascending=False
)

results_df

results_df.to_csv(
    "../results/baseline_models.csv",
    index=False
)

In [37]:
results_long = results_df.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Score"
)

fig = px.bar(
    results_long,
    x="Metric",
    y="Score",
    color="Model",
    barmode="group",
    title="Baseline Model Comparison"
)

fig.write_image("../screenshots/baseline_models.png")
fig.show()

## Cross Validation

In [38]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [39]:
cv_results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Mean ROC AUC": scores.mean(),
        "Std": scores.std()
    })

In [40]:
cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by="Mean ROC AUC",
    ascending=False
)

cv_results_df

,Model,Mean ROC AUC,Std
3,Gradient Boosting,0.846444,0.004688
0,Logistic Regression,0.845020,0.002642
2,Random Forest,0.819436,0.002918
1,Decision Tree,0.656528,0.019211


In [41]:
cv_results_df.to_csv(
    "../results/cross_validation.csv",
    index=False
)

In [42]:
fig = px.bar(
    cv_results_df,
    x="Model",
    y="Mean ROC AUC",
    error_y="Std",
    title="Cross Validation Performance"
)

fig.write_image("../screenshots/cross_validation.png")
fig.show()

## Hyperparameter Tuning (GridSearchCV)

In [43]:
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_depth": [2, 3, 4],
    "classifier__subsample": [0.8, 1.0]
}

In [44]:
grid_search = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

In [45]:
grid_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__learning_rate': [0.05, 0.1], 'classifier__max_depth': [2, 3, ...], 'classifier__n_estimators': [100, 200], 'classifier__subsample': [0.8, 1.0]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know

In [46]:
grid_search.best_params_

{'classifier__learning_rate': 0.05,
 'classifier__max_depth': 3,
 'classifier__n_estimators': 100,
 'classifier__subsample': 0.8}

In [47]:
grid_search.best_score_

np.float64(0.8502024504603725)

In [48]:
best_model = grid_search.best_estimator_

In [49]:
best_predictions = best_model.predict(X_test)

best_probabilities = best_model.predict_proba(X_test)[:, 1]

In [50]:
tuned_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC AUC"
    ],
    "Baseline": [
        accuracy_score(y_test, models["Gradient Boosting"].predict(X_test)),
        precision_score(y_test, models["Gradient Boosting"].predict(X_test)),
        recall_score(y_test, models["Gradient Boosting"].predict(X_test)),
        f1_score(y_test, models["Gradient Boosting"].predict(X_test)),
        roc_auc_score(
            y_test,
            models["Gradient Boosting"].predict_proba(X_test)[:,1]
        )
    ],
    "Tuned": [
        accuracy_score(y_test, best_predictions),
        precision_score(y_test, best_predictions),
        recall_score(y_test, best_predictions),
        f1_score(y_test, best_predictions),
        roc_auc_score(y_test, best_probabilities)
    ]
})

tuned_results

tuned_results.to_csv(
    "../results/tuned_vs_baseline.csv",
    index=False
)

In [51]:
comparison = tuned_results.melt(
    id_vars="Metric",
    var_name="Model",
    value_name="Score"
)

fig = px.bar(
    comparison,
    x="Metric",
    y="Score",
    color="Model",
    barmode="group",
    title="Baseline vs Tuned Gradient Boosting"
)

fig.write_image(
    "../screenshots/tuned_vs_baseline.png"
)
fig.show()

## RandomizedSearchCV

In [52]:
param_dist = {
    "classifier__n_estimators": [100, 150, 200, 250, 300],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "classifier__max_depth": [2, 3, 4, 5],
    "classifier__subsample": [0.6, 0.8, 1.0]
}

In [53]:
random_search = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

In [54]:
random_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__learning_rate': [0.01, 0.03, ...], 'classifier__max_depth': [2, 3, ...], 'classifier__n_estimators': [100, 150, ...], 'classifier__subsample': [0.6, 0.8, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score i

In [55]:
random_search.best_params_

{'classifier__subsample': 1.0,
 'classifier__n_estimators': 100,
 'classifier__max_depth': 2,
 'classifier__learning_rate': 0.1}

In [56]:
random_search.best_score_

np.float64(0.8498963453642894)

In [57]:
best_random = random_search.best_estimator_

In [58]:
random_predictions = best_random.predict(X_test)

random_probabilities = best_random.predict_proba(X_test)[:,1]

In [59]:
random_results = pd.DataFrame({
    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC AUC"
    ],
    "Score":[
        accuracy_score(y_test, random_predictions),
        precision_score(y_test, random_predictions),
        recall_score(y_test, random_predictions),
        f1_score(y_test, random_predictions),
        roc_auc_score(y_test, random_probabilities)
    ]
})

random_results

,Metric,Score
0,Accuracy,0.792466
1,Precision,0.638514
2,Recall,0.505348
3,F1 Score,0.564179
4,ROC AUC,0.839284


In [60]:
random_results.to_csv(
    "../results/random_search_results.csv",
    index=False
)

In [61]:
comparison = pd.DataFrame({
    "Method":[
        "Baseline",
        "Grid Search",
        "Random Search"
    ],
    "ROC AUC":[
        0.8386144918233069,
        roc_auc_score(y_test, best_probabilities),
        roc_auc_score(y_test, random_probabilities)
    ]
})

comparison

,Method,ROC AUC
0,Baseline,0.838614
1,Grid Search,0.839350
2,Random Search,0.839284


In [62]:
fig = px.bar(
    comparison,
    x="Method",
    y="ROC AUC",
    color="Method",
    title="Hyperparameter Optimization Comparison"
)

fig.write_image("../screenshots/hyperparameter_comparison.png")
fig.show()

## Feature Selection (SelectKBest)

In [63]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [64]:
selector = SelectKBest(
    score_func=mutual_info_classif,
    k=20
)

X_train_selected = selector.fit_transform(
    X_train_processed,
    y_train
)

X_test_selected = selector.transform(
    X_test_processed
)

In [65]:
feature_names = preprocessor.get_feature_names_out()

selected_features = pd.DataFrame({
    "Feature": feature_names[selector.get_support()],
    "Score": selector.scores_[selector.get_support()]
})

selected_features = selected_features.sort_values(
    by="Score",
    ascending=False
)

selected_features

,Feature,Score
15,cat__Contract_Month-to-month,0.099193
0,num__tenure,0.070066
17,cat__Contract_Two year,0.069685
5,cat__OnlineSecurity_No,0.053675
11,cat__TechSupport_No,0.053046
1,num__MonthlyCharges,0.048571
2,num__TotalCharges,0.045442
3,cat__InternetService_Fiber optic,0.042200
6,cat__OnlineSecurity_No internet service,0.041829
10,cat__DeviceProtection_No internet service,0.041686


In [66]:
selected_features.to_csv(
    "../results/selectkbest_features.csv",
    index=False
)

In [67]:
fig = px.bar(
    selected_features.head(20),
    x="Score",
    y="Feature",
    orientation="h",
    title="Top Features (SelectKBest)"
)

fig.write_image("../screenshots/selectkbest_features.png")
fig.show()

Recursive Feature Elimination (RFE)

In [68]:
rfe_estimator = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [69]:
rfe = RFE(
    estimator=rfe_estimator,
    n_features_to_select=20
)

In [70]:
rfe.fit(
    X_train_processed,
    y_train
)

,"estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance(e.g. `coef_`, `feature_importances_`).",LogisticRegre...ndom_state=42)
,"n_features_to_select n_features_to_select: int or float, default=NoneThe number of features to select. If `None`, half of the features areselected. If integer, the parameter is the absolute number of featuresto select. If float between 0 and 1, it is the fraction of features toselect... versionchanged:: 0.24 Added float values for fractions.",20
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.",1
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance (implemented with `attrgetter`).For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case ofclass:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only available when `estimator` is a classifier.","ndarray[int64](2,)","[0,1]"
estimator_ estimator_: ``Estimator`` instanceThe fitted estimator used to select features.,LogisticRegression,LogisticRegre...ndom_state=42)
n_features_ n_features_: intThe number of selected features.,int64,np.int64(20)
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,45
"ranking_ ranking_: ndarray of shape (n_features,)The feature ranking, such that ``ranking_[i]`` corresponds to theranking position of the i-th feature. Selected (i.e., estimatedbest) features are assigned rank 1.","ndarray[int64](45,)","[22, 1, 1,...,14, 1,15]"


In [71]:
rfe_features = pd.DataFrame({
    "Feature": feature_names,
    "Selected": rfe.support_,
    "Ranking": rfe.ranking_
})

rfe_features = rfe_features.sort_values(
    by="Ranking"
)

rfe_features.head(25)

,Feature,Selected,Ranking
1,num__tenure,True,1
2,num__MonthlyCharges,True,1
3,num__TotalCharges,True,1
15,cat__InternetService_DSL,True,1
12,cat__MultipleLines_No,True,1
19,cat__OnlineSecurity_No internet service,True,1
28,cat__TechSupport_No internet service,True,1
31,cat__StreamingTV_No internet service,True,1
25,cat__DeviceProtection_No internet service,True,1
30,cat__StreamingTV_No,True,1


In [72]:
rfe_features.to_csv(
    "../results/rfe_features.csv",
    index=False
)

In [73]:
top_rfe = rfe_features.head(20)

fig = px.bar(
    top_rfe,
    x="Ranking",
    y="Feature",
    orientation="h",
    title="Top Features Selected by RFE"
)

fig.write_image(
    "../screenshots/rfe_features.png"
)
fig.show()

## Learning Curves

In [74]:
train_sizes, train_scores, validation_scores = learning_curve(
    estimator=best_model,
    X=X,
    y=y,
    cv=cv,
    scoring="roc_auc",
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1,
    shuffle=True,
    random_state=42
)

In [75]:
train_mean = train_scores.mean(axis=1)
validation_mean = validation_scores.mean(axis=1)

In [76]:
learning_df = pd.DataFrame({
    "Training Size": train_sizes,
    "Training Score": train_mean,
    "Validation Score": validation_mean
})

learning_df

,Training Size,Training Score,Validation Score
0,562,0.952725,0.828257
1,1125,0.922906,0.838216
2,1687,0.904702,0.842470
3,2250,0.894652,0.844046
4,2812,0.884168,0.846056
5,3375,0.879197,0.848039
6,3937,0.876475,0.846624
7,4500,0.872384,0.847424
8,5062,0.870358,0.847217
9,5625,0.868886,0.848129


In [77]:
learning_df.to_csv(
    "../results/learning_curve.csv",
    index=False
)

In [78]:
fig = px.line(
    learning_df,
    x="Training Size",
    y=["Training Score", "Validation Score"],
    markers=True,
    title="Learning Curve"
)

fig.write_image(
    "../screenshots/learning_curve.png"
)
fig.show()

## Validation Curve

In [79]:
depth_range = [1, 2, 3, 4, 5, 6]

In [80]:
train_scores, validation_scores = validation_curve(
    estimator=gb_pipeline,
    X=X,
    y=y,
    param_name="classifier__max_depth",
    param_range=depth_range,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

In [81]:
train_mean = train_scores.mean(axis=1)
validation_mean = validation_scores.mean(axis=1)

In [82]:
validation_df = pd.DataFrame({
    "Max Depth": depth_range,
    "Training Score": train_mean,
    "Validation Score": validation_mean
})

validation_df

,Max Depth,Training Score,Validation Score
0,1,0.851730,0.846913
1,2,0.863661,0.847675
2,3,0.881696,0.846444
3,4,0.906662,0.845356
4,5,0.937593,0.840181
5,6,0.967294,0.835006


In [83]:
validation_df.to_csv(
    "../results/validation_curve.csv",
    index=False
)

In [84]:
fig = px.line(
    validation_df,
    x="Max Depth",
    y=["Training Score", "Validation Score"],
    markers=True,
    title="Validation Curve (Max Depth)"
)

fig.write_image(
    "../screenshots/validation_curve.png"
)
fig.show()

## Permutation Importance

In [85]:
permutation = permutation_importance(
    estimator=best_model,
    X=X_test,
    y=y_test,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc",
    n_jobs=-1
)

In [87]:
permutation_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": permutation.importances_mean
})

permutation_df = permutation_df.sort_values(
    by="Importance",
    ascending=False
)

permutation_df.head(20)

,Feature,Importance
14,Contract,0.081268
4,tenure,0.043734
7,InternetService,0.016365
17,MonthlyCharges,0.011220
18,TotalCharges,0.006438
11,TechSupport,0.005418
8,OnlineSecurity,0.004879
16,PaymentMethod,0.003907
15,PaperlessBilling,0.003096
13,StreamingMovies,0.001184


In [88]:
permutation_df.to_csv(
    "../results/permutation_importance.csv",
    index=False
)

In [89]:
fig = px.bar(
    permutation_df.head(20),
    x="Importance",
    y="Feature",
    orientation="h",
    title="Permutation Feature Importance"
)

fig.write_image(
    "../screenshots/permutation_importance.png"
)
fig.show()

## Error Analysis

In [90]:
error_df = X_test.copy()

error_df["Actual"] = y_test.values

error_df["Predicted"] = best_predictions

error_df["Probability"] = best_probabilities

In [91]:
error_df["Result"] = np.where(
    error_df["Actual"] == error_df["Predicted"],
    "Correct",
    "Incorrect"
)

error_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Actual,Predicted,Probability,Result
974,Female,0,Yes,Yes,59,Yes,No,DSL,No,Yes,...,Yes,Two year,Yes,Credit card (automatic),75.95,4542.35,0,0,0.034591,Correct
619,Female,0,No,No,7,Yes,Yes,Fiber optic,No,Yes,...,No,Month-to-month,Yes,Bank transfer (automatic),78.55,522.95,0,1,0.652208,Incorrect
4289,Female,0,No,No,54,Yes,No,No,No internet service,No internet service,...,No internet service,Two year,No,Mailed check,20.10,1079.45,0,0,0.026011,Correct
3721,Female,0,No,No,2,Yes,No,No,No internet service,No internet service,...,No internet service,Month-to-month,No,Mailed check,20.65,38.70,1,0,0.155059,Incorrect
4533,Female,0,Yes,No,71,Yes,Yes,Fiber optic,No,Yes,...,Yes,Two year,Yes,Bank transfer (automatic),105.15,7555.00,0,0,0.108850,Correct


In [92]:
misclassified = error_df[
    error_df["Result"] == "Incorrect"
]

misclassified.head(20)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Actual,Predicted,Probability,Result
619,Female,0,No,No,7,Yes,Yes,Fiber optic,No,Yes,...,No,Month-to-month,Yes,Bank transfer (automatic),78.55,522.95,0,1,0.652208,Incorrect
3721,Female,0,No,No,2,Yes,No,No,No internet service,No internet service,...,No internet service,Month-to-month,No,Mailed check,20.65,38.70,1,0,0.155059,Incorrect
445,Female,0,No,No,60,Yes,Yes,Fiber optic,No,Yes,...,Yes,Month-to-month,Yes,Electronic check,105.90,6396.45,1,0,0.465668,Incorrect
4283,Male,1,Yes,No,4,Yes,No,Fiber optic,No,No,...,No,Month-to-month,Yes,Electronic check,70.20,280.35,0,1,0.644489,Incorrect
6128,Female,0,Yes,No,14,Yes,No,Fiber optic,No,No,...,Yes,Month-to-month,Yes,Electronic check,78.10,1122.40,0,1,0.628951,Incorrect
4653,Female,0,Yes,Yes,30,No,No phone service,DSL,No,No,...,Yes,Month-to-month,No,Credit card (automatic),51.20,1561.50,1,0,0.130263,Incorrect
2300,Female,0,Yes,Yes,48,Yes,Yes,Fiber optic,Yes,Yes,...,Yes,One year,Yes,Credit card (automatic),103.25,5037.55,1,0,0.173548,Incorrect
6344,Female,1,Yes,No,38,Yes,Yes,Fiber optic,No,No,...,No,Month-to-month,Yes,Bank transfer (automatic),74.95,2869.85,1,0,0.366006,Incorrect
978,Male,1,Yes,No,62,Yes,Yes,Fiber optic,No,Yes,...,Yes,One year,Yes,Electronic check,103.75,6383.35,1,0,0.319764,Incorrect
2968,Male,0,No,No,3,Yes,No,Fiber optic,No,Yes,...,Yes,Month-to-month,No,Electronic check,90.40,268.45,0,1,0.642910,Incorrect


In [93]:
misclassified.to_csv(
    "../results/misclassified_samples.csv",
    index=False
)

In [94]:
false_positive = error_df[
    (error_df["Actual"] == 0) &
    (error_df["Predicted"] == 1)
]

false_positive.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Actual,Predicted,Probability,Result
619,Female,0,No,No,7,Yes,Yes,Fiber optic,No,Yes,...,No,Month-to-month,Yes,Bank transfer (automatic),78.55,522.95,0,1,0.652208,Incorrect
4283,Male,1,Yes,No,4,Yes,No,Fiber optic,No,No,...,No,Month-to-month,Yes,Electronic check,70.20,280.35,0,1,0.644489,Incorrect
6128,Female,0,Yes,No,14,Yes,No,Fiber optic,No,No,...,Yes,Month-to-month,Yes,Electronic check,78.10,1122.40,0,1,0.628951,Incorrect
2968,Male,0,No,No,3,Yes,No,Fiber optic,No,Yes,...,Yes,Month-to-month,No,Electronic check,90.40,268.45,0,1,0.642910,Incorrect
3612,Male,0,No,No,21,Yes,Yes,Fiber optic,No,No,...,No,Month-to-month,Yes,Electronic check,73.70,1558.70,0,1,0.561439,Incorrect


In [95]:
false_negative = error_df[
    (error_df["Actual"] == 1) &
    (error_df["Predicted"] == 0)
]

false_negative.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Actual,Predicted,Probability,Result
3721,Female,0,No,No,2,Yes,No,No,No internet service,No internet service,...,No internet service,Month-to-month,No,Mailed check,20.65,38.70,1,0,0.155059,Incorrect
445,Female,0,No,No,60,Yes,Yes,Fiber optic,No,Yes,...,Yes,Month-to-month,Yes,Electronic check,105.90,6396.45,1,0,0.465668,Incorrect
4653,Female,0,Yes,Yes,30,No,No phone service,DSL,No,No,...,Yes,Month-to-month,No,Credit card (automatic),51.20,1561.50,1,0,0.130263,Incorrect
2300,Female,0,Yes,Yes,48,Yes,Yes,Fiber optic,Yes,Yes,...,Yes,One year,Yes,Credit card (automatic),103.25,5037.55,1,0,0.173548,Incorrect
6344,Female,1,Yes,No,38,Yes,Yes,Fiber optic,No,No,...,No,Month-to-month,Yes,Bank transfer (automatic),74.95,2869.85,1,0,0.366006,Incorrect


In [96]:
error_summary = pd.DataFrame({
    "Type":[
        "Correct",
        "Incorrect",
        "False Positive",
        "False Negative"
    ],
    "Count":[
        len(error_df)-len(misclassified),
        len(misclassified),
        len(false_positive),
        len(false_negative)
    ]
})

error_summary

,Type,Count
0,Correct,1122
1,Incorrect,285
2,False Positive,106
3,False Negative,179


In [97]:
error_summary.to_csv(
    "../results/error_summary.csv",
    index=False
)

In [98]:
fig = px.bar(
    error_summary,
    x="Type",
    y="Count",
    color="Type",
    title="Prediction Error Analysis"
)

fig.show()

In [99]:
fig.write_image(
    "../screenshots/error_analysis.png"
)

## Save Optimized Pipeline

In [100]:
joblib.dump(
    best_model,
    "../models/best_model.pkl"
)

['../models/best_model.pkl']

In [101]:
joblib.dump(
    best_model.named_steps["preprocessor"],
    "../models/preprocessor.pkl"
)

['../models/preprocessor.pkl']

In [102]:
joblib.dump(
    selector,
    "../models/feature_selector.pkl"
)

['../models/feature_selector.pkl']